<a href="https://colab.research.google.com/github/MohamedX86/flyrank-ml-internship-2026/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohamedX86/flyrank-ml-internship-2026/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content item for one client on one report date.

I use the fact_content_daily_performance table for March 2026, covering 2026-03-01 through 2026-03-31.

My prediction target is future content performance, using historical content metrics as input features.

I deliberately exclude the final month, June 2026, because it must remain a sealed test window.

I also exclude any label-derived or future-looking fields from the honest feature set.

In [29]:
!pip -q install duckdb huggingface_hub

import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN was not found in Colab Secrets."

os.environ["HF_TOKEN"] = HF_TOKEN

print("Setup complete. HF token loaded safely.")

Setup complete. HF token loaded safely.


In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [31]:
import duckdb

con = duckdb.connect()

con.sql("""
INSTALL httpfs;
LOAD httpfs;
""")

print("DuckDB Ready")

DuckDB Ready


In [32]:
con.sql(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("Hugging Face access connected to DuckDB.")

Hugging Face access connected to DuckDB.


In [33]:
result = con.sql("""
SELECT COUNT(*) AS row_count
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

result

,row_count
0,9841378


### Verification Query 1 – Grain

This query verifies that each row is unique for the combination of:

- client_hash_id
- content_hash_id
- report_date

Result:
- Total rows: 9,841,378
- Distinct grain rows: 9,841,378

This confirms that one row represents one content item for one client on one report date.

In [34]:
grain_check = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(
        DISTINCT client_hash_id || '|' ||
        content_hash_id || '|' ||
        CAST(report_date AS VARCHAR)
    ) AS distinct_grain_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_rows
0,9841378,9841378


### Verification Query 2 – Date Window

This query verifies the date range used for this assignment.

Result:

- Start date: 2026-03-01
- End date: 2026-03-31
- Total rows: 9,841,378

In [35]:
date_range = con.sql("""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS row_count
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

date_range

,start_date,end_date,row_count
0,2026-03-01,2026-03-31,9841378


### Verification Query 3 – Data Availability

This query checks how many rows have both Google Search Console and Google Analytics data available.

Condition:

- gsc_data_available IS TRUE
- ga4_data_available IS TRUE

Result:

364,347 rows satisfy both conditions.

In [36]:
availability = con.sql("""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE
    gsc_data_available IS TRUE
    AND ga4_data_available IS TRUE
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,364347


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions

### Label
- Future content performance (prediction target)

### Context
- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4

### Excluded
- Future-looking columns or any label-derived fields (to prevent data leakage)
- June 2026 data (kept as the final unseen test window)

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [39]:
feature_frame = con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE
    gsc_data_available IS TRUE
    AND ga4_data_available IS TRUE
LIMIT 1000
""").df()

feature_frame.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,5,0,5.400000,1,1
1,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,39,0,5.666667,2,2
2,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,179,0,5.156425,2,2
3,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,72,0,7.694444,1,1
4,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,3282,1,6.167885,1,1


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limitations

- This dataset only covers the available warehouse data for the selected month.
- Not every row has both GSC and GA4 data available.
- June 2026 is excluded to keep the final evaluation unbiased.
- Historical metrics cannot capture future events or external factors that may affect content performance.

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Five Features

| Feature | Available when? | Why use it? |
|---------|-----------------|-------------|
| gsc_impressions | Available on the report date when GSC data exists. | Measures how often the content appeared in Google Search results. |
| gsc_clicks | Available on the report date when GSC data exists. | Measures how many search users clicked the content. |
| gsc_avg_position | Available on the report date when GSC data exists. | Represents the average ranking position in Google Search. |
| ga4_pageviews | Available on the report date when GA4 data exists. | Measures page traffic from Google Analytics 4. |
| ga4_sessions | Available on the report date when GA4 data exists. | Measures user sessions for the content. |

In [41]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# Copy the feature frame so the original stays unchanged
leakage_df = feature_frame.copy()

# Create a simple binary label:
# 1 = pageviews above the median, 0 = pageviews at or below the median
pageview_median = leakage_df["ga4_pageviews"].median()
leakage_df["high_future_performance"] = (
    leakage_df["ga4_pageviews"] > pageview_median
).astype(int)

# Honest features: exclude ga4_pageviews because the label was derived from it
honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
]

X_honest = leakage_df[honest_features].fillna(0)
y = leakage_df["high_future_performance"]

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

honest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
)

honest_model.fit(X_train, y_train)
honest_predictions = honest_model.predict(X_test)
honest_accuracy = accuracy_score(y_test, honest_predictions)

# Deliberate leakage:
# copy the label directly into a feature column
leakage_df["leaked_label_feature"] = leakage_df["high_future_performance"]

leaked_features = honest_features + ["leaked_label_feature"]
X_leaked = leakage_df[leaked_features].fillna(0)

X_train_leaked, X_test_leaked, y_train_leaked, y_test_leaked = train_test_split(
    X_leaked,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

leaked_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
)

leaked_model.fit(X_train_leaked, y_train_leaked)
leaked_predictions = leaked_model.predict(X_test_leaked)
leaked_accuracy = accuracy_score(y_test_leaked, leaked_predictions)

print(f"Honest accuracy: {honest_accuracy:.3f}")
print(f"Leaked accuracy: {leaked_accuracy:.3f}")

# Remove the leaked column after the experiment
leakage_df = leakage_df.drop(columns=["leaked_label_feature"])

print("Leaked feature removed:", "leaked_label_feature" not in leakage_df.columns)

Honest accuracy: 0.860
Leaked accuracy: 1.000
Leaked feature removed: True


### Deliberate Leakage Experiment

I first trained a model using only features that were available at the decision moment.

- Honest accuracy: 0.860

I then added a feature copied directly from the target label.

- Leaked accuracy: 1.000

The perfect score was artificial because the model was given the answer inside the input data. I removed the leaked feature after the experiment and kept the honest score of 0.860.

This confirms that label-derived and future-looking columns must be excluded from the final feature set.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.